<a href="https://colab.research.google.com/github/AnanyaUp/ANN-Deep-Learning/blob/main/Optimizers_and_Regularization_for_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Hyperparameters
input_size = 28 * 28
hidden_sizes = [512, 256]
num_classes = 10
num_epochs = 10
batch_size = 64
learning_rate = 0.001
weight_decay = 1e-4
dropout_rate = 0.5

In [ ]:
# Load dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.13,), (0.30,))
])

In [ ]:
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

In [ ]:
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Neural network with Dropout regularization
class DeepNN(nn.Module):
    def __init__(self, input_size, hidden_sizes, num_classes, dropout_rate):
        super(DeepNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_sizes[0])
        self.drop1 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_sizes[0], hidden_sizes[1])
        self.drop2 = nn.Dropout(dropout_rate)
        self.fc3 = nn.Linear(hidden_sizes[1], num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, input_size)
        x = self.relu(self.fc1(x))
        x = self.drop1(x)
        x = self.relu(self.fc2(x))
        x = self.drop2(x)
        x = self.fc3(x)
        return x

In [ ]:
model = DeepNN(input_size, hidden_sizes, num_classes, dropout_rate).to(device)

In [ ]:
# Loss and optimizer(SGD)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

In [ ]:
# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/10], Loss: 2.1611
Epoch [2/10], Loss: 1.6704
Epoch [3/10], Loss: 1.1433
Epoch [4/10], Loss: 0.8723
Epoch [5/10], Loss: 0.7404
Epoch [6/10], Loss: 0.6562
Epoch [7/10], Loss: 0.5994
Epoch [8/10], Loss: 0.5603
Epoch [9/10], Loss: 0.5308
Epoch [10/10], Loss: 0.5031


In [ ]:
# Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test Accuracy: {100 * correct / total:.2f}%')

Test Accuracy: 90.21%
